<img src = '../images/pokemon.jpg'>

### Pokemon - Data Wrangling
* Physical Cards, English Only, Secondary Market Price - Retail
* There will be nulls from the source data randomly in extCardType, marketPrice, and subTypeName.
    * Less than 1.7% of total cards at the most.

In [1]:
import pandas as pd
import numpy as np
import re

#### Define clean datetimes function.

In [2]:
# The purpose of this function is to convert errant release dates in source group data to a format pd.datetime can recognize, so it can convert to a datetime object.
# pd.datetime cannot convert after 7 decimals.
# For reference:
# https://docs.python.org/3/library/re.html
# re.sub(pattern, repl, string, count=0, flags=0)
def cleanAndParseDates(dateStr):
    if pd.isna(dateStr):
        return pd.NaT
    
    # Trim after 6 digits, r'\1' references the first capture group in the first ().
    dateStr = re.sub(r'(\.\d{6})\d*Z$', r'\1', dateStr)

    # Remove Z (UTC indicator) if still present.
    dateStr = re.sub(r'Z$', '', dateStr)

    try:
        # Parsing datetimes with mixed time zones will raise an error unless utc=True.
        return pd.to_datetime(dateStr, utc=True)
    
    except Exception:
        return pd.NaT

#### Clean card data.

In [3]:
# Source data date 10/15/25
dfp = pd.read_excel('../data/dataPokemon/cardsPokemon.xlsx')

# Standardize column name.
dfp = dfp.rename(columns = {"Source.Name" : "sourceName"})

# Specify the needed columns.
# extNumber will be used to identify single cards vs all other products (basic energy cards, boosters, code cards, figurines, decks, tins, etc.).
dfp = dfp[["sourceName", "productId", "cleanName", "groupId", "extNumber", "extRarity", "extCardType", "marketPrice", "subTypeName"]]

# Product without an extNumber needs to be removed from the dataframe.
# First convert null to NaN then drop.
dfp['extNumber'].replace('', np.nan, inplace=True)
dfp.dropna(subset=['extNumber'], inplace = True)

C:\Users\arsta\AppData\Local\Temp\ipykernel_61100\3076520691.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfp['extNumber'].replace('', np.nan, inplace=True)


#### Clean set group information.

In [4]:
dfpGroups = pd.read_csv('../data/dataPokemon/groupsPokemon.csv')

dfpGroups = dfpGroups[["groupId", "name", "abbreviation", "publishedOn"]]

dfpGroups = dfpGroups.rename(columns = {"name" : "groupName", "abbreviation" : "groupCode", "publishedOn" : "releaseDate"})

# There are consistently 4 NaN Group Codes from origin data. 
# Since codes are universal and static once established, it is relatively safe to fill them in using precedent.
dfpGroups.loc[20, 'groupCode'] = 'MYFB'
dfpGroups.loc[31, 'groupCode'] = 'PPSC'
dfpGroups.loc[123, 'groupCode'] = 'MCD12'
dfpGroups.loc[129, 'groupCode'] = 'MCD11'

# Applying the function defined above.
dfpGroups['releaseDate'] = dfpGroups['releaseDate'].apply(cleanAndParseDates)

# Drop timezone (make native datetime) and normalize to remove time info.
dfpGroups['releaseDate'] = dfpGroups['releaseDate'].dt.tz_convert(None).dt.normalize()

# Create and insert a releaseYear column for plotting later.
dfpGroups['releaseYear'] = dfpGroups['releaseDate'].dt.year

#### Merge cards with set groups.

In [5]:
dfp2 = pd.merge(dfp, dfpGroups, on = "groupId", how = "inner")

# Drop sourceName now that we have groupName.
dfp2.drop(columns = ["sourceName"], inplace = True)

# A more viewer-friendly order:
newOrderP = ['cleanName', 'groupId', 'groupName', 'productId', 'subTypeName', 'extCardType', 'extNumber', 'extRarity', 'releaseDate', 'releaseYear', 'marketPrice']
dfp2 = dfp2[newOrderP]

# Order by releaseDate, then groupName, then cleanName.
dfp2 = dfp2.sort_values(by=["releaseDate", "groupName", "cleanName"])

# Reset index after manipulation and to check new number of rows.
# Dropping the original index column.
dfp2 = dfp2.reset_index(drop = True)

# If needed as its own csv file, uncomment:
dfp2.to_csv("../data/dataPokemon/cleanPokemon.csv", index = False)

# Make a pickle file if CSV file size is too large.
# dfp2.to_pickle("../data/dataPokemon/dfp2.pkl")

### Search check to ensure functionality.

In [6]:
dfp2[dfp2["cleanName"] == "Snorlax"]

,cleanName,groupId,groupName,productId,subTypeName,extCardType,extNumber,extRarity,releaseDate,releaseYear,marketPrice
623,Snorlax,1418.0,WoTC Promo,89385.0,Normal,Colorless,49/53,Promo,1999-07-01,1999,177.20
865,Snorlax,605.0,Base Set 2,42502.0,Normal,Colorless,030/130,Rare,2000-02-24,2000,9.71
2519,Snorlax,1374.0,Legendary Collection,89386.0,Normal,Colorless,064/110,Uncommon,2002-05-24,2002,14.29
2520,Snorlax,1374.0,Legendary Collection,89386.0,Reverse Holofoil,Colorless,064/110,Uncommon,2002-05-24,2002,300.00
3464,Snorlax,1372.0,Skyridge,89387.0,Normal,Colorless,100/144,Common,2003-05-12,2003,69.49
4691,Snorlax,1419.0,FireRed & LeafGreen,89388.0,Normal,Colorless,15/112,Holo Rare,2004-08-30,2004,85.00
4692,Snorlax,1419.0,FireRed & LeafGreen,89388.0,Holofoil,Colorless,15/112,Holo Rare,2004-08-30,2004,119.14
4693,Snorlax,1419.0,FireRed & LeafGreen,89388.0,Reverse Holofoil,Colorless,15/112,Holo Rare,2004-08-30,2004,109.98
7125,Snorlax,1430.0,Diamond and Pearl,89389.0,Normal,Colorless,37/130,Rare,2007-05-23,2007,8.14
7126,Snorlax,1430.0,Diamond and Pearl,89389.0,Reverse Holofoil,Colorless,37/130,Rare,2007-05-23,2007,54.34
